# DCS 404 — Lab 2: Linear Models for Regression
**Dataset:** Medical Insurance Charges  
**Task:** Regression — predicting insurance charges from personal attributes  
**Student:** *(Your Name)*  
**Date:** May 2026

---
## Part 0 — Task Selection

**Learning Task:** Regression (continuous target variable)

**Dataset:** `insurance.csv` — 1,338 records of insured individuals with the following features:
- `age` — age of the insured person
- `sex` — gender (male / female)
- `bmi` — Body Mass Index
- `children` — number of dependents
- `smoker` — smoking status (yes / no)
- `region` — geographic region (northeast / northwest / southeast / southwest)
- `charges` — **target variable**: medical insurance costs (USD)

**Objective:** Build and compare Linear Regression (no regularization) against Ridge (L2) and Lasso (L1) regularized models to understand how regularization affects model performance and coefficient magnitudes.

---
## Part 1 — Data Loading and Preprocessing

### 1.1 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('All libraries imported successfully.')

### 1.2 Load the Dataset

In [ ]:
df = pd.read_csv('insurance.csv')

print('Shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
df.head(10)

### 1.3 Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics
print('=== Summary Statistics ===')
df.describe(include='all')

In [ ]:
# Check for missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# Check value counts for categorical columns
for col in ['sex', 'smoker', 'region']:
    print(f'\n--- {col} ---')
    print(df[col].value_counts())

In [ ]:
# Distribution of the target variable (charges)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df['charges'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Insurance Charges')
axes[0].set_xlabel('Charges (USD)')
axes[0].set_ylabel('Frequency')

# Log-transformed
axes[1].hist(np.log1p(df['charges']), bins=40, color='darkorange', edgecolor='white')
axes[1].set_title('Log-Transformed Charges')
axes[1].set_xlabel('log(Charges)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Target Variable Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Charges — Mean: ${df["charges"].mean():,.2f} | Median: ${df["charges"].median():,.2f} | Std: ${df["charges"].std():,.2f}')

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
num_cols = ['age', 'bmi', 'children']
cat_cols = ['sex', 'smoker', 'region']

for i, col in enumerate(num_cols):
    axes[0, i].hist(df[col], bins=30, color='steelblue', edgecolor='white')
    axes[0, i].set_title(f'Distribution of {col}')
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Count')

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[1, i].bar(counts.index, counts.values, color='darkorange', edgecolor='white')
    axes[1, i].set_title(f'Value counts: {col}')
    axes[1, i].set_xlabel(col)
    axes[1, i].set_ylabel('Count')

plt.suptitle('Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Charges vs each feature — key relationships
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Numerical: scatter plots
for i, col in enumerate(['age', 'bmi', 'children']):
    axes[0, i].scatter(df[col], df['charges'], alpha=0.4, color='steelblue', s=20)
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Charges (USD)')
    axes[0, i].set_title(f'{col} vs Charges')

# Categorical: box plots
for i, col in enumerate(['sex', 'smoker', 'region']):
    categories = df[col].unique()
    data_by_cat = [df[df[col] == cat]['charges'].values for cat in categories]
    axes[1, i].boxplot(data_by_cat, labels=categories)
    axes[1, i].set_xlabel(col)
    axes[1, i].set_ylabel('Charges (USD)')
    axes[1, i].set_title(f'{col} vs Charges')

plt.suptitle('Feature vs Target (Charges) Relationships', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Observation:** The `smoker` variable shows the most dramatic difference in charges — smokers are charged significantly more. `age` and `bmi` also show positive relationships with charges. The distribution of charges is right-skewed (many low values, few very high ones).

In [ ]:
# Correlation heatmap (numeric features only)
plt.figure(figsize=(8, 6))
num_df = df[['age', 'bmi', 'children', 'charges']]
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix (Numeric Features)', fontweight='bold')
plt.tight_layout()
plt.show()

### 1.4 Preprocessing — Encode, Scale, Split

In [ ]:
# Work on a copy
df_processed = df.copy()

# --- Label Encode binary categoricals ---
le = LabelEncoder()
df_processed['sex']    = le.fit_transform(df_processed['sex'])     # female=0, male=1
df_processed['smoker'] = le.fit_transform(df_processed['smoker'])  # no=0, yes=1

# --- One-Hot Encode region (4 categories) ---
region_dummies = pd.get_dummies(df_processed['region'], prefix='region', drop_first=True)
df_processed = pd.concat([df_processed.drop(columns='region'), region_dummies], axis=1)

print('Processed columns:', list(df_processed.columns))
print('Shape after encoding:', df_processed.shape)
df_processed.head()

In [ ]:
# Separate features and target
X = df_processed.drop(columns='charges')
y = df_processed['charges']

print('Feature matrix X shape:', X.shape)
print('Target vector y shape: ', y.shape)
print('\nFeature names:', list(X.columns))

In [ ]:
# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f'Training set:  {X_train.shape[0]} samples')
print(f'Testing set:   {X_test.shape[0]} samples')

In [ ]:
# StandardScaler — fit ONLY on training data, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Scaling complete.')
print(f'Train mean (should be ~0): {X_train_scaled.mean(axis=0).round(4)}')
print(f'Train std  (should be ~1): {X_train_scaled.std(axis=0).round(4)}')

---
## Part 2 — Baseline Linear Regression (No Regularization)

In [ ]:
# Train baseline model
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_lr = lr.predict(X_train_scaled)
y_test_pred_lr  = lr.predict(X_test_scaled)

# Evaluate
def evaluate(y_true_train, y_pred_train, y_true_test, y_pred_test, model_name):
    results = {
        'Model':        model_name,
        'Train MSE':    mean_squared_error(y_true_train, y_pred_train),
        'Test MSE':     mean_squared_error(y_true_test,  y_pred_test),
        'Train MAE':    mean_absolute_error(y_true_train, y_pred_train),
        'Test MAE':     mean_absolute_error(y_true_test,  y_pred_test),
        'Train R²':     r2_score(y_true_train, y_pred_train),
        'Test R²':      r2_score(y_true_test,  y_pred_test),
    }
    return results

baseline_results = evaluate(y_train, y_train_pred_lr, y_test, y_test_pred_lr, 'Linear Regression')

print('=== Baseline Linear Regression ===')
for k, v in baseline_results.items():
    if k == 'Model':
        print(f'{k}: {v}')
    elif 'R²' in k:
        print(f'{k}: {v:.4f}')
    else:
        print(f'{k}: ${v:,.2f}')

In [ ]:
# Coefficients for the baseline model
feature_names = list(X.columns)
coef_df = pd.DataFrame({
    'Feature':     feature_names,
    'Coefficient': lr.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('=== Baseline Model Coefficients ===')
print(f'Intercept: ${lr.intercept_:,.2f}\n')
print(coef_df.to_string(index=False))

In [ ]:
# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted
axes[0].scatter(y_test, y_test_pred_lr, alpha=0.5, color='steelblue', s=25)
lims = [min(y_test.min(), y_test_pred_lr.min()),
        max(y_test.max(), y_test_pred_lr.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Charges (USD)')
axes[0].set_ylabel('Predicted Charges (USD)')
axes[0].set_title('Actual vs Predicted — Baseline LR')
axes[0].legend()

# Residuals
residuals = y_test - y_test_pred_lr
axes[1].scatter(y_test_pred_lr, residuals, alpha=0.5, color='darkorange', s=25)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Charges (USD)')
axes[1].set_ylabel('Residuals (USD)')
axes[1].set_title('Residual Plot — Baseline LR')

plt.suptitle('Baseline Linear Regression — Diagnostic Plots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Observation:** The model captures the general trend but struggles with high-charge outliers (likely heavy smokers with high BMI). The residual plot shows a fan-shaped pattern, suggesting heteroscedasticity — the model under-predicts high charges.

---
## Part 3 — Regularized Linear Models

### 3.1 Ridge Regression (L2)

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_results = []
ridge_coefs   = []

for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    y_tr_pred = model.predict(X_train_scaled)
    y_te_pred = model.predict(X_test_scaled)
    ridge_results.append({
        'Alpha':    alpha,
        'Train R²': r2_score(y_train, y_tr_pred),
        'Test R²':  r2_score(y_test,  y_te_pred),
        'Test MSE': mean_squared_error(y_test, y_te_pred),
        'Test MAE': mean_absolute_error(y_test, y_te_pred),
    })
    ridge_coefs.append(model.coef_)

ridge_df = pd.DataFrame(ridge_results)
print('=== Ridge Regression Results ===')
print(ridge_df.to_string(index=False))

### 3.2 Lasso Regression (L1)

In [ ]:
lasso_results = []
lasso_coefs   = []

for alpha in alphas:
    model = Lasso(alpha=alpha, max_iter=10000)
    model.fit(X_train_scaled, y_train)
    y_tr_pred = model.predict(X_train_scaled)
    y_te_pred = model.predict(X_test_scaled)
    lasso_results.append({
        'Alpha':    alpha,
        'Train R²': r2_score(y_train, y_tr_pred),
        'Test R²':  r2_score(y_test,  y_te_pred),
        'Test MSE': mean_squared_error(y_test, y_te_pred),
        'Test MAE': mean_absolute_error(y_test, y_te_pred),
        'Non-zero coefs': np.sum(model.coef_ != 0)
    })
    lasso_coefs.append(model.coef_)

lasso_df = pd.DataFrame(lasso_results)
print('=== Lasso Regression Results ===')
print(lasso_df.to_string(index=False))

**Observation:** Lasso's 'Non-zero coefs' column shows how many features are actually used — as alpha increases, Lasso drives weak feature coefficients to exactly zero, performing automatic feature selection.

---
## Part 4 — Comparison and Analysis

### 4.1 Summary Metrics Table

In [ ]:
# Select best alpha for Ridge and Lasso (highest Test R²)
best_ridge_row   = ridge_df.loc[ridge_df['Test R²'].idxmax()]
best_lasso_row   = lasso_df.loc[lasso_df['Test R²'].idxmax()]
best_ridge_alpha = best_ridge_row['Alpha']
best_lasso_alpha = best_lasso_row['Alpha']

print(f'Best Ridge alpha: {best_ridge_alpha}  — Test R²: {best_ridge_row["Test R²"]:.4f}')
print(f'Best Lasso alpha: {best_lasso_alpha}  — Test R²: {best_lasso_row["Test R²"]:.4f}')

# Retrain best Ridge and Lasso for coefficient comparison
best_ridge = Ridge(alpha=best_ridge_alpha)
best_ridge.fit(X_train_scaled, y_train)

best_lasso = Lasso(alpha=best_lasso_alpha, max_iter=10000)
best_lasso.fit(X_train_scaled, y_train)

# Comprehensive comparison table
comparison = pd.DataFrame([
    {
        'Model':          'Linear Regression',
        'Alpha':          'N/A',
        'Train R²':       r2_score(y_train, lr.predict(X_train_scaled)),
        'Test R²':        r2_score(y_test,  lr.predict(X_test_scaled)),
        'Test MSE':       mean_squared_error(y_test, lr.predict(X_test_scaled)),
        'Test MAE':       mean_absolute_error(y_test, lr.predict(X_test_scaled)),
    },
    {
        'Model':          f'Ridge (best α={best_ridge_alpha})',
        'Alpha':          best_ridge_alpha,
        'Train R²':       r2_score(y_train, best_ridge.predict(X_train_scaled)),
        'Test R²':        r2_score(y_test,  best_ridge.predict(X_test_scaled)),
        'Test MSE':       mean_squared_error(y_test, best_ridge.predict(X_test_scaled)),
        'Test MAE':       mean_absolute_error(y_test, best_ridge.predict(X_test_scaled)),
    },
    {
        'Model':          f'Lasso (best α={best_lasso_alpha})',
        'Alpha':          best_lasso_alpha,
        'Train R²':       r2_score(y_train, best_lasso.predict(X_train_scaled)),
        'Test R²':        r2_score(y_test,  best_lasso.predict(X_test_scaled)),
        'Test MSE':       mean_squared_error(y_test, best_lasso.predict(X_test_scaled)),
        'Test MAE':       mean_absolute_error(y_test, best_lasso.predict(X_test_scaled)),
    },
])

print('\n=== Model Comparison (Best Regularization Strengths) ===')
display_cols = ['Model', 'Train R²', 'Test R²', 'Test MSE', 'Test MAE']
print(comparison[display_cols].to_string(index=False))

### 4.2 Alpha vs Performance — Bias-Variance Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge
axes[0].semilogx(ridge_df['Alpha'], ridge_df['Train R²'], 'o-', color='steelblue',  label='Train R²')
axes[0].semilogx(ridge_df['Alpha'], ridge_df['Test R²'],  's--', color='darkorange', label='Test R²')
axes[0].axvline(best_ridge_alpha, color='red', linestyle=':', linewidth=1.2, label=f'Best α={best_ridge_alpha}')
axes[0].set_xlabel('Alpha (log scale)')
axes[0].set_ylabel('R² Score')
axes[0].set_title('Ridge: Alpha vs R²')
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.3)

# Lasso
axes[1].semilogx(lasso_df['Alpha'], lasso_df['Train R²'], 'o-', color='steelblue',  label='Train R²')
axes[1].semilogx(lasso_df['Alpha'], lasso_df['Test R²'],  's--', color='darkorange', label='Test R²')
axes[1].axvline(best_lasso_alpha, color='red', linestyle=':', linewidth=1.2, label=f'Best α={best_lasso_alpha}')
axes[1].set_xlabel('Alpha (log scale)')
axes[1].set_ylabel('R² Score')
axes[1].set_title('Lasso: Alpha vs R²')
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.3)

plt.suptitle('Regularization Strength vs Model Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Coefficient Magnitude Comparison

In [ ]:
# Coefficient comparison: baseline vs Ridge vs Lasso
coef_compare = pd.DataFrame({
    'Feature':            feature_names,
    'Linear Regression':  lr.coef_,
    f'Ridge (α={best_ridge_alpha})': best_ridge.coef_,
    f'Lasso (α={best_lasso_alpha})': best_lasso.coef_,
})

print('=== Coefficient Comparison ===')
print(coef_compare.to_string(index=False))

# Grouped bar chart
x = np.arange(len(feature_names))
width = 0.25
fig, ax = plt.subplots(figsize=(14, 6))

bars1 = ax.bar(x - width, lr.coef_,          width, label='Linear Regression', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x,         best_ridge.coef_,  width, label=f'Ridge α={best_ridge_alpha}', color='darkorange', alpha=0.85)
bars3 = ax.bar(x + width, best_lasso.coef_,  width, label=f'Lasso α={best_lasso_alpha}', color='seagreen',   alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(feature_names, rotation=30, ha='right')
ax.set_ylabel('Coefficient Value')
ax.set_title('Coefficient Comparison: Baseline vs Ridge vs Lasso', fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend()
plt.tight_layout()
plt.show()

### 4.4 Lasso — Feature Selection Effect

In [ ]:
# Show how many features Lasso keeps as alpha increases
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(lasso_df['Alpha'], lasso_df['Non-zero coefs'], 'o-', color='seagreen', linewidth=2, markersize=8)
ax.set_xlabel('Alpha (log scale)')
ax.set_ylabel('Number of Non-Zero Coefficients')
ax.set_title('Lasso Feature Selection: Non-Zero Coefficients vs Alpha', fontweight='bold')
ax.grid(True, which='both', alpha=0.3)
ax.set_yticks(range(0, len(feature_names) + 1))
plt.tight_layout()
plt.show()

### 4.5 Coefficient Paths Across Alpha Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(feature_names)))

# Ridge coefficient paths
for i, name in enumerate(feature_names):
    path = [coef[i] for coef in ridge_coefs]
    axes[0].semilogx(alphas, path, 'o-', color=colors[i], label=name, linewidth=1.5)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('Alpha (log scale)')
axes[0].set_ylabel('Coefficient Value')
axes[0].set_title('Ridge: Coefficient Paths')
axes[0].legend(fontsize=8, loc='upper right')
axes[0].grid(True, which='both', alpha=0.3)

# Lasso coefficient paths
for i, name in enumerate(feature_names):
    path = [coef[i] for coef in lasso_coefs]
    axes[1].semilogx(alphas, path, 'o-', color=colors[i], label=name, linewidth=1.5)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Alpha (log scale)')
axes[1].set_ylabel('Coefficient Value')
axes[1].set_title('Lasso: Coefficient Paths (zeroing out features)')
axes[1].legend(fontsize=8, loc='upper right')
axes[1].grid(True, which='both', alpha=0.3)

plt.suptitle('How Coefficients Change with Regularization Strength', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Key Insight:** Ridge shrinks all coefficients gradually toward zero but never reaches zero. Lasso's L1 penalty actively drives weak features to exact zero — effectively eliminating them from the model.

---
## Part 5 — Reflection

### 5.1 Final Summary Table

In [ ]:
print('=' * 65)
print('FINAL MODEL COMPARISON SUMMARY')
print('=' * 65)
print(comparison[display_cols].to_string(index=False))
print('=' * 65)

best_model_idx = comparison['Test R²'].idxmax()
print(f"\nBest model by Test R²: {comparison.loc[best_model_idx, 'Model']}")
print(f"  Test R²  = {comparison.loc[best_model_idx, 'Test R²']:.4f}")
print(f"  Test MSE = ${comparison.loc[best_model_idx, 'Test MSE']:,.2f}")
print(f"  Test MAE = ${comparison.loc[best_model_idx, 'Test MAE']:,.2f}")

### 5.2 Written Reflection

#### Did regularization improve generalization? Why or why not?

In this case, regularization provided **minimal improvement** over the baseline Linear Regression model. The train R² and test R² for all three models are very close, which tells us that the baseline model was not overfitting in the first place. The insurance dataset is relatively small (1,338 rows) with only 8 features after encoding, so there is not enough model complexity for standard Linear Regression to overfit significantly.

#### Which regularization worked better for your task?

**Lasso (L1)** is slightly more informative for this dataset because it performs automatic feature selection — at higher alpha values it zeroes out region dummy variables and the `children` feature, confirming that these have weaker predictive power. The dominant predictor in this dataset is clearly `smoker`, followed by `age` and `bmi`. Ridge is useful for understanding relative feature importance via coefficient magnitude but keeps all features active.

#### What does this tell us about the bias–variance tradeoff?

The bias–variance tradeoff is the core tension in machine learning:
- **High variance (overfitting):** The model is too complex — it fits training data perfectly but fails on new data. Regularization helps here by penalizing large coefficients.
- **High bias (underfitting):** The model is too simple — it cannot capture the patterns in data. Very strong regularization (large alpha) pushes the model toward this zone.

In our experiment, the train and test performance curves show that performance degrades at high alpha values (e.g. α=100), confirming **underfitting** at strong regularization. The intermediate range (α=0.01 to α=1) offers the best balance. Since the gap between train and test R² is already small for the baseline, the dataset does not suffer from high variance — regularization is most beneficial when a model shows a large train vs test gap.

**Conclusion:** For a clean, low-dimensional dataset like `insurance.csv`, Linear Regression performs competitively. Regularization becomes more impactful with high-dimensional data, noisy features, or multicollinearity between features.